<a href="https://colab.research.google.com/github/aakilkhanemon/AI-driven-marine-lead-discovery/blob/main/rec3_x_remdsim_success.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import math
import shutil
import subprocess
import numpy as np
import pandas as pd
from rdkit import Chem

print("==========================================================")
print("🔑 INITIALIZING SECURE CLOUD STORAGE SYNC: RECEPTOR-3")
print("==========================================================")

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Absolute paths mapped exactly to your visible sidebar layout
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_LEADS_DIR = f"{DRIVE_PROJECT_DIR}/Leads_remdsim_Top_5"

RECEPTOR = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"
INPUT_LIBRARY = f"{DRIVE_PROJECT_DIR}/remdsim.sdf"
TARGET_RESIDUES_STR = "resid 41 80 108 119 120 134"

BATCH_SIZE = 15
CONTACT_CUTOFF = 4.0

# Install dependencies locally if missing
try:
    import MDAnalysis as mda
except ImportError:
    subprocess.run("pip install -q rdkit mdanalysis pandas", shell=True)
    import MDAnalysis as mda

# Pull down the deep learning driver binary locally
if not os.path.exists("gnina"):
    print("• Transferring binary architecture to runtime engine...")
    subprocess.run("wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina", shell=True)
subprocess.run("chmod +x gnina", shell=True)

# 🛡️ FIXED: Verifying the absolute Google Drive paths
if not os.path.exists(RECEPTOR) or not os.path.exists(INPUT_LIBRARY):
    raise FileNotFoundError(f"❌ Error: Files not found! Verify receptor3.pdb and remdsim.sdf are inside your Drive folder: {DRIVE_PROJECT_DIR}")

# ==========================================
# 2. DYNAMIC CAVITY GEOMETRY MAPPING
# ==========================================
print("\n➔ Mapping 3D spatial boundaries from target amino acid network...")
u_rec = mda.Universe(RECEPTOR)
pocket_atoms = u_rec.select_atoms(TARGET_RESIDUES_STR)

if len(pocket_atoms) == 0:
    raise ValueError("❌ Selection Mapping Failure: Targeted residues could not be resolved!")

CX, CY, CZ = pocket_atoms.center_of_mass()
coordinate_span = np.max(pocket_atoms.positions, axis=0) - np.min(pocket_atoms.positions, axis=0)
BOX_SIZE = max(16, math.ceil(np.max(coordinate_span) + 10.0))

print(f"   • Active Coordinates: X={CX:.3f} | Y={CY:.3f} | Z={CZ:.3f} (Box: {BOX_SIZE}Å)")

# ==========================================
# 3. STATE-AWARE RESUME DOCKING PIPELINE
# ==========================================
supplier = Chem.SDMolSupplier(INPUT_LIBRARY)
molecules = [m for m in supplier if m is not None]
total_mols = len(molecules)
total_batches = math.ceil(total_mols / BATCH_SIZE)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_LEADS_DIR, exist_ok=True)

print(f"\n➔ Commencing virtual screen loop over {total_mols} molecules in remdsim.sdf...")

for b in range(total_batches):
    batch_sdf = f"{CHECKPOINT_DIR}/batch_{b+1}_input.sdf"
    batch_out = f"{CHECKPOINT_DIR}/batch_{b+1}_docked.sdf"

    # Skip instantly if the output file is already safe in your Drive
    if os.path.exists(batch_out) and os.path.getsize(batch_out) > 0:
        print(f"⏩ Batch {b+1}/{total_batches} previously stabilized in Drive. Bypassing.")
        continue

    print(f"🚀 Computing Checkpoint Batch {b+1}/{total_batches}...")

    writer = Chem.SDWriter(batch_sdf)
    for m in molecules[b*BATCH_SIZE : (b+1)*BATCH_SIZE]:
        writer.write(m)
    writer.close()

    gnina_cmd = (
        f"./gnina -r {RECEPTOR} -l {batch_sdf} "
        f"--center_x {CX} --center_y {CY} --center_z {CZ} "
        f"--size_x {BOX_SIZE} --size_y {BOX_SIZE} --size_z {BOX_SIZE} "
        f"--num_modes 1 --exhaustiveness 1 --device 0 "
        f"--out {batch_out} --cnn_scoring rescore"
    )
    subprocess.run(gnina_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ Complete structural data sync stabilized permanently in your Google Drive.")

# ==========================================
# 4. POST-TRAJECTORY MATRIX EXTRACTOR
# ==========================================
print("\n➔ Processing residue-specific distance filtration matrix from cloud checkpoints...")
target_coords = pocket_atoms.positions
valid_site_binders = []
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for file_path in docked_files:
    supp = Chem.SDMolSupplier(file_path)
    for mol in supp:
        if mol is None: continue
        conf = mol.GetConformer()
        ligand_coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
        distances = np.linalg.norm(ligand_coords[:, np.newaxis, :] - target_coords[np.newaxis, :, :], axis=2)

        if np.min(distances) <= CONTACT_CUTOFF:
            mol_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(valid_site_binders)}"
            valid_site_binders.append({
                "mol_obj": mol,
                "Identifier": mol_name,
                "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
                "CNN_Score": float(mol.GetProp('CNNscore')),
                "CNN_Affinity": float(mol.GetProp('CNNaffinity')),
                "Proximity_Å": round(np.min(distances), 3)
            })

if len(valid_site_binders) == 0:
    print("❌ Selection Matrix Failure: Zero compounds docked inside the active pocket envelope.")
else:
    df_filtered = pd.DataFrame(valid_site_binders)
    df_filtered = df_filtered.sort_values(by="Vina_Energy", ascending=True).reset_index(drop=True)
    top_5_leads = df_filtered.head(5)

    with open(RECEPTOR, 'r') as f:
        receptor_text = f.read()

    for idx, row in top_5_leads.iterrows():
        clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
        with open(f"{OUTPUT_LEADS_DIR}/Absolute_Rank_{idx+1}_{clean_id}_complex.pdb", 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

    print("\n==========================================================")
    print(" 🏆 FINAL PROJECT SITE LEAD DISCOVERY BOARD (TOP 5)")
    print("==========================================================")
    print(top_5_leads[["Identifier", "Vina_Energy", "CNN_Score", "CNN_Affinity", "Proximity_Å"]].to_string(index=False))
    print("==========================================================")
    print(f"🎉 Process Complete. Files are safe in your Google Drive: '{DRIVE_PROJECT_DIR}'")

🔑 INITIALIZING SECURE CLOUD STORAGE SYNC: RECEPTOR-3
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

➔ Mapping 3D spatial boundaries from target amino acid network...
   • Active Coordinates: X=-14.826 | Y=-20.885 | Z=-21.064 (Box: 25Å)

➔ Commencing virtual screen loop over 301 molecules in remdsim.sdf...
⏩ Batch 1/21 previously stabilized in Drive. Bypassing.
🚀 Computing Checkpoint Batch 2/21...
🚀 Computing Checkpoint Batch 3/21...
🚀 Computing Checkpoint Batch 4/21...


In [2]:
# 1. Package acceleration suite
!pip install -q rdkit mdanalysis pandas

# 2. Binary compilation retrieval
!wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina
!chmod +x gnina

print("==========================================================")
print("✅ NEW NOTEBOOK STORAGE ENVIRONMENT VERIFIED")
print("==========================================================")
print("• Receptor Template Target : receptor3.pdb")
print("• Active Screening Library : remdsim.sdf")
print("• GNINA Neural Driver     : Live")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.1/37.1 MB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 2.1 MB/s eta 0:00:00
✅ NEW NOTEBOOK STORAGE ENVIRONMENT VERIFIED
• Receptor Template Target : receptor3.pdb
• Active Screening Library : remdsim.sdf
• GNINA Neural Driver     : Live


In [ ]:
import os
import sys
import math
import shutil
import subprocess
import numpy as np
import pandas as pd
from rdkit import Chem

print("==========================================================")
print("🔑 RESUMING DISCONNECTED RECEPTOR-3 SCREENING ARCHITECTURE")
print("==========================================================")

# 1. Verify Cloud Sync Connection
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except Exception:
    drive.mount('/content/drive')

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
LOCAL_SCRATCH = "/content/scratch_resubmit"

# Enforce explicit pathway anchors based on your structural history
RECEPTOR = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"
INPUT_LIBRARY = f"{DRIVE_PROJECT_DIR}/remdsim.sdf"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_LEADS_DIR = f"{DRIVE_PROJECT_DIR}/Leads_remdsim_Top_5"

# HARDCODED SPATIAL KEYS RECOVERED FROM YOUR INTERRUPTED CONSOLE RECORD
CX, CY, CZ = -14.826, -20.885, -21.064
BOX_SIZE = 25
TARGET_RESIDUES_STR = "resid 41 80 108 119 120 134"
CONTACT_CUTOFF = 4.0

# UPDATED BLOCK PARSING INDEX
BATCH_SIZE = 15

try:
    import MDAnalysis as mda
except ImportError:
    subprocess.run("pip install -q rdkit mdanalysis pandas", shell=True)
    import MDAnalysis as mda

os.makedirs(LOCAL_SCRATCH, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_LEADS_DIR, exist_ok=True)

# Deploy local binary compute frame to maximize acceleration efficiency
if not os.path.exists("gnina"):
    print("• Deploying native binary processing architecture...")
    subprocess.run("wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina", shell=True)
subprocess.run("chmod +x gnina", shell=True)

if not os.path.exists(RECEPTOR) or not os.path.exists(INPUT_LIBRARY):
    raise FileNotFoundError(f"❌ Verification Error: Ensure your source files are inside: {DRIVE_PROJECT_DIR}")

# ==========================================
# 2. EVALUATE CHEMICAL MATRIX SLICES
# ==========================================
supplier = Chem.SDMolSupplier(INPUT_LIBRARY)
molecules = [m for m in supplier if m is not None]
total_mols = len(molecules)
total_batches = math.ceil(total_mols / BATCH_SIZE)

print(f"\n➔ Commencing virtual screen loop over {total_mols} molecules in remdsim.sdf...")

for b in range(total_batches):
    batch_sdf = f"{CHECKPOINT_DIR}/batch_{b+1}_input.sdf"
    batch_out = f"{CHECKPOINT_DIR}/batch_{b+1}_docked.sdf"

    # 🛡️ RESUME GAURDRAIL: Skip if the file exists on Drive AND is not an empty/interrupted remnant
    if os.path.exists(batch_out) and os.path.getsize(batch_out) > 100:
        print(f"⏩ Batch {b+1}/{total_batches} previously stabilized in Drive. Bypassing execution.")
        continue

    print(f"🚀 Computing Checkpoint Batch {b+1}/{total_batches}...")

    # Isolate sub-batch to local fast storage disk space
    writer = Chem.SDWriter(batch_sdf)
    for m in molecules[b*BATCH_SIZE : (b+1)*BATCH_SIZE]:
        writer.write(m)
    writer.close()

    gnina_cmd = (
        f"./gnina -r {RECEPTOR} -l {batch_sdf} "
        f"--center_x {CX} --center_y {CY} --center_z {CZ} "
        f"--size_x {BOX_SIZE} --size_y {BOX_SIZE} --size_z {BOX_SIZE} "
        f"--num_modes 1 --exhaustiveness 1 --device 0 "
        f"--out {batch_out} --cnn_scoring rescore"
    )
    subprocess.run(gnina_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("✅ Complete trajectory data sync stabilized permanently in your Google Drive.")

# ==========================================
# 3. POST-TRAJECTORY MATRIX FILTRATION
# ==========================================
print("\n➔ Processing residue-specific distance filtration matrix from checkpoints...")
u_rec = mda.Universe(RECEPTOR)
pocket_atoms = u_rec.select_atoms(TARGET_RESIDUES_STR)
target_coords = pocket_atoms.positions

valid_site_binders = []
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for file_path in docked_files:
    supp = Chem.SDMolSupplier(file_path)
    for mol in supp:
        if mol is None: continue
        conf = mol.GetConformer()
        ligand_coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
        distances = np.linalg.norm(ligand_coords[:, np.newaxis, :] - target_coords[np.newaxis, :, :], axis=2)

        if np.min(distances) <= CONTACT_CUTOFF:
            mol_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(valid_site_binders)}"
            valid_site_binders.append({
                "mol_obj": mol,
                "Identifier": mol_name,
                "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
                "CNN_Score": float(mol.GetProp('CNNscore')),
                "CNN_Affinity": float(mol.GetProp('CNNaffinity')),
                "Proximity_Å": round(np.min(distances), 3)
            })

if len(valid_site_binders) == 0:
    print("❌ Selection Matrix Failure: Zero compounds managed to coordinate within 4.0 Å of target residues.")
else:
    df_filtered = pd.DataFrame(valid_site_binders)
    # Sort strictly ascending to prioritize the lowest/tightest thermodynamic values
    df_filtered = df_filtered.sort_values(by="Vina_Energy", ascending=True).reset_index(drop=True)
    top_5_leads = df_filtered.head(5)

    with open(RECEPTOR, 'r') as f:
        receptor_text = f.read()

    for idx, row in top_5_leads.iterrows():
        clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
        output_pdb = f"{OUTPUT_LEADS_DIR}/Absolute_Rank_{idx+1}_{clean_id}_complex.pdb"
        with open(output_pdb, 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

    print("\n==========================================================")
    print(" 🏆 RESUMED RUN: CORE MECHANISTIC TARGET LEAD MATRIX (TOP 5)")
    print("==========================================================")
    print(top_5_leads[["Identifier", "Vina_Energy", "CNN_Score", "CNN_Affinity", "Proximity_Å"]].to_string(index=False))
    print("==========================================================")
    print(f"🎉 Process Complete. Files are safe in your Google Drive: '{DRIVE_PROJECT_DIR}'")

🔑 RESUMING DISCONNECTED RECEPTOR-3 SCREENING ARCHITECTURE
Mounted at /content/drive

➔ Commencing virtual screen loop over 301 molecules in remdsim.sdf...
⏩ Batch 1/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 2/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 3/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 4/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 5/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 6/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 7/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 8/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 9/21 previously stabilized in Drive. Bypassing execution.
⏩ Batch 10/21 previously stabilized in Drive. Bypassing execution.
🚀 Computing Checkpoint Batch 11/21...
🚀 Computing Checkpoint Batch 12/21...
🚀 Computing Checkpoint Batch 13/21...
🚀 Computing Checkpoint Batch 14/21...
🚀 Computing Checkpoint Batch 15/

[07:44:07] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[07:44:07] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[07:44:07] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[07:44:07] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[07:44:07] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[07:44:07] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[07:44:07] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[07:44:08] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[07:44:08] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

KeyError: 'minimizedAffinity'

In [ ]:
import os
import sys
import math
import shutil
import subprocess
import numpy as np
import pandas as pd
from rdkit import Chem

print("==========================================================")
print("⚡ DEPLOYING BULLETPROOF LOCAL-WRITE RESUME TRAJECTORY")
print("==========================================================")

# 1. Mount and verify remote cloud socket
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except Exception:
    drive.mount('/content/drive')

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
LOCAL_SCRATCH = "/content/scratch_safe_zone"

RECEPTOR = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"
INPUT_LIBRARY = f"{DRIVE_PROJECT_DIR}/remdsim.sdf"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_LEADS_DIR = f"{DRIVE_PROJECT_DIR}/Leads_remdsim_Top_5"

# Hardcoded coordinates matching your exact pocket history
CX, CY, CZ = -14.826, -20.885, -21.064
BOX_SIZE = 25
TARGET_RESIDUES_STR = "resid 41 80 108 119 120 134"
CONTACT_CUTOFF = 4.0
BATCH_SIZE = 15

try:
    import MDAnalysis as mda
except ImportError:
    subprocess.run("pip install -q rdkit mdanalysis pandas", shell=True)
    import MDAnalysis as mda

os.makedirs(LOCAL_SCRATCH, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_LEADS_DIR, exist_ok=True)

if not os.path.exists("gnina"):
    subprocess.run("wget -q https://github.com/gnina/gnina/releases/download/v1.1/gnina -O gnina", shell=True)
subprocess.run("chmod +x gnina", shell=True)

if not os.path.exists(RECEPTOR) or not os.path.exists(INPUT_LIBRARY):
    raise FileNotFoundError(f"❌ Storage Error: Missing input assets inside: {DRIVE_PROJECT_DIR}")

# ==========================================
# 2. RUNTIME ACCELERATED RESUME ENGINE
# ==========================================
supplier = Chem.SDMolSupplier(INPUT_LIBRARY)
molecules = [m for m in supplier if m is not None]
total_mols = len(molecules)
total_batches = math.ceil(total_mols / BATCH_SIZE)

print(f"➔ Resuming screening across {total_mols} molecules (Split into {total_batches} batches)...")

for b in range(total_batches):
    batch_sdf_local = f"{LOCAL_SCRATCH}/batch_{b+1}_input.sdf"
    batch_out_local = f"{LOCAL_SCRATCH}/batch_{b+1}_docked.sdf"
    batch_out_drive = f"{CHECKPOINT_DIR}/batch_{b+1}_docked.sdf"

    # Checkpoint Guard: Skip if the batch is already safe on your Drive
    if os.path.exists(batch_out_drive) and os.path.getsize(batch_out_drive) > 100:
        print(f"⏩ Batch {b+1}/{total_batches} is safe on Drive. Skipping.")
        continue

    print(f"🚀 Processing Batch {b+1}/{total_batches} entirely in local scratch memory...")

    # Write input slice to fast local drive space
    writer = Chem.SDWriter(batch_sdf_local)
    for m in molecules[b*BATCH_SIZE : (b+1)*BATCH_SIZE]:
        writer.write(m)
    writer.close()

    # FIXED: GNINA now outputs to local scratch instead of Drive path
    gnina_cmd = (
        f"./gnina -r {RECEPTOR} -l {batch_sdf_local} "
        f"--center_x {CX} --center_y {CY} --center_z {CZ} "
        f"--size_x {BOX_SIZE} --size_y {BOX_SIZE} --size_z {BOX_SIZE} "
        f"--num_modes 1 --exhaustiveness 2 --device 0 "
        f"--out {batch_out_local} --cnn_scoring rescore"
    )
    subprocess.run(gnina_cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Safe Sync Step: Transfer the completed local file to Google Drive in a single move
    if os.path.exists(batch_out_local) and os.path.getsize(batch_out_local) > 0:
        shutil.copy2(batch_out_local, batch_out_drive)
        # Clear temporary local storage files
        os.remove(batch_sdf_local)
        os.remove(batch_out_local)

print("✅ Screening loop complete. Disconnection protection successful.")

# ==========================================
# 3. LEAD EXTRACTION SELECTION PIPELINE
# ==========================================
print("\n➔ Running atomistic proximity filtering across completed trajectories...")
u_rec = mda.Universe(RECEPTOR)
pocket_atoms = u_rec.select_atoms(TARGET_RESIDUES_STR)
target_coords = pocket_atoms.positions

valid_site_binders = []
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for file_path in docked_files:
    supp = Chem.SDMolSupplier(file_path)
    for mol in supp:
        if mol is None: continue
        conf = mol.GetConformer()
        ligand_coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
        distances = np.linalg.norm(ligand_coords[:, np.newaxis, :] - target_coords[np.newaxis, :, :], axis=2)

        if np.min(distances) <= CONTACT_CUTOFF:
            mol_name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(valid_site_binders)}"
            valid_site_binders.append({
                "mol_obj": mol,
                "Identifier": mol_name,
                "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
                "CNN_Score": float(mol.GetProp('CNNscore')),
                "CNN_Affinity": float(mol.GetProp('CNNaffinity')),
                "Proximity_Å": round(np.min(distances), 3)
            })

if len(valid_site_binders) == 0:
    print("❌ Selection Matrix Failure: Zero compounds mapped within the pocket boundary.")
else:
    df_filtered = pd.DataFrame(valid_site_binders)
    df_filtered = df_filtered.sort_values(by="Vina_Energy", ascending=True).reset_index(drop=True)
    top_5_leads = df_filtered.head(5)

    with open(RECEPTOR, 'r') as f:
        receptor_text = f.read()

    for idx, row in top_5_leads.iterrows():
        clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
        output_pdb = f"{OUTPUT_LEADS_DIR}/Absolute_Rank_{idx+1}_{clean_id}_complex.pdb"
        with open(output_pdb, 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

    print("\n==========================================================")
    print(" 🏆 TARGET LEAD DISCOVERY MATRIX COMPILATION (TOP 5)")
    print("==========================================================")
    print(top_5_leads[["Identifier", "Vina_Energy", "CNN_Score", "CNN_Affinity", "Proximity_Å"]].to_string(index=False))
    print("==========================================================")
    print(f"🎉 Complete. Output complex files saved to Drive path: '{OUTPUT_LEADS_DIR}'")

⚡ DEPLOYING BULLETPROOF LOCAL-WRITE RESUME TRAJECTORY
Mounted at /content/drive
➔ Resuming screening across 301 molecules (Split into 21 batches)...
⏩ Batch 1/21 is safe on Drive. Skipping.
⏩ Batch 2/21 is safe on Drive. Skipping.
⏩ Batch 3/21 is safe on Drive. Skipping.
⏩ Batch 4/21 is safe on Drive. Skipping.
⏩ Batch 5/21 is safe on Drive. Skipping.
⏩ Batch 6/21 is safe on Drive. Skipping.
⏩ Batch 7/21 is safe on Drive. Skipping.
⏩ Batch 8/21 is safe on Drive. Skipping.
⏩ Batch 9/21 is safe on Drive. Skipping.
⏩ Batch 10/21 is safe on Drive. Skipping.
🚀 Processing Batch 11/21 entirely in local scratch memory...


KeyboardInterrupt: 

In [3]:
import os
import math
import pandas as pd
from rdkit import Chem
from google.colab import drive

print("==========================================================")
print("📊 SCRIPT 1: LIGAND EXTRACTION BY VINA THERMODYNAMIC SCORE")
print("==========================================================")

# Mount Google Drive
try: drive.mount('/content/drive')
except Exception: pass

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/Top_5_Percent_Vina"

RECEPTOR_PATH = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"

if not os.path.exists(CHECKPOINT_DIR):
    raise FileNotFoundError(f"❌ Checkpoint directory missing at: {CHECKPOINT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_poses = []
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for f_path in docked_files:
    supp = Chem.SDMolSupplier(f_path)
    for mol in supp:
        if mol is None: continue
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(all_poses)}"
        all_poses.append({
            "mol_obj": mol,
            "Identifier": name,
            "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
            "CNN_Score": float(mol.GetProp('CNNscore')),
            "CNN_Affinity": float(mol.GetProp('CNNaffinity'))
        })

df = pd.DataFrame(all_poses)
# Sort ascending: lowest/most negative free energy values represent premium binders
df_sorted = df.sort_values(by="Vina_Energy", ascending=True).reset_index(drop=True)

cutoff = max(1, int(math.ceil(len(df_sorted) * 0.05)))
top_vina = df_sorted.head(cutoff)

print(f"\n🏆 TOP 5% VINA SCORE LEADERBOARD (Isolating {cutoff} out of {len(df_sorted)} targets)")
print("-" * 70)
print(top_vina[["Identifier", "Vina_Energy", "CNN_Score"]].to_string(index=False))
print("-" * 70)

with open(RECEPTOR_PATH, 'r') as f:
    receptor_text = f.read()

for idx, row in top_vina.iterrows():
    clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
    out_pdb = f"{OUTPUT_DIR}/Vina_Rank_{idx+1}_{clean_id}_complex.pdb"
    with open(out_pdb, 'w') as out:
        out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

print(f"\n🎉 Success! Top 5% Vina complexes exported to: {OUTPUT_DIR}")

📊 SCRIPT 1: LIGAND EXTRACTION BY VINA THERMODYNAMIC SCORE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:02:17] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

KeyError: 'minimizedAffinity'

In [4]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from google.colab import drive

print("==========================================================")
print("🔍 REFINED MECHANISM-SPECIFIC RESIDUE INTERACTION FILTER")
print("==========================================================")

try: drive.mount('/content/drive')
except Exception: pass

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/Mechanism_Specific_Top_5"

RECEPTOR_PATH = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"
TARGET_RESIDUES_STR = "resid 41 80 108 119 120 134"
CONTACT_CUTOFF = 4.0

try:
    import MDAnalysis as mda
except ImportError:
    import subprocess
    subprocess.run("pip install -q rdkit mdanalysis pandas", shell=True)
    import MDAnalysis as mda

if not os.path.exists(CHECKPOINT_DIR):
    raise FileNotFoundError(f"❌ Checkpoint directory missing at: {CHECKPOINT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

u_rec = mda.Universe(RECEPTOR_PATH)
pocket_atoms = u_rec.select_atoms(TARGET_RESIDUES_STR)
target_coords = pocket_atoms.positions

valid_binders = []
skipped_count = 0
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for f_path in docked_files:
    supp = Chem.SDMolSupplier(f_path)
    for mol in supp:
        if mol is None: continue

        # 🛡️ PROTECTIVE LAYER: Skip molecules that lack GNINA metadata tags
        if not mol.HasProp('minimizedAffinity') or not mol.HasProp('CNNscore'):
            skipped_count += 1
            continue

        conf = mol.GetConformer()
        ligand_coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
        distances = np.linalg.norm(ligand_coords[:, np.newaxis, :] - target_coords[np.newaxis, :, :], axis=2)
        min_dist = np.min(distances)

        if min_dist <= CONTACT_CUTOFF:
            name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(valid_binders)}"
            valid_binders.append({
                "mol_obj": mol,
                "Identifier": name,
                "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
                "CNN_Score": float(mol.GetProp('CNNscore')),
                "CNN_Affinity": float(mol.GetProp('CNNaffinity')),
                "Proximity_Å": round(min_dist, 3)
            })

print(f"• Trajectory scan complete. (Safely bypassed {skipped_count} un-docked anomalies).")

if len(valid_binders) == 0:
    print("❌ Selection Matrix Failure: Zero compounds located within 4.0 Å of active site side chains.")
else:
    df_filtered = pd.DataFrame(valid_binders)
    df_sorted = df_filtered.sort_values(by="Vina_Energy", ascending=True).reset_index(drop=True)
    top_5_leads = df_sorted.head(5)

    print("\n🏆 ABSOLUTE MECHANISTIC POCKET LEADERBOARD (TOP 5 LEADS)")
    print("-" * 85)
    print(top_5_leads[["Identifier", "Vina_Energy", "CNN_Score", "CNN_Affinity", "Proximity_Å"]].to_string(index=False))
    print("-" * 85)

    with open(RECEPTOR_PATH, 'r') as f:
        receptor_text = f.read()

    for idx, row in top_5_leads.iterrows():
        clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
        out_pdb = f"{OUTPUT_DIR}/Absolute_Mechanistic_Rank_{idx+1}_{clean_id}_complex.pdb"
        with open(out_pdb, 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

    print(f"\n🎉 Success! Top 5 mechanism-specific complexes exported to: {OUTPUT_DIR}")

🔍 REFINED MECHANISM-SPECIFIC RESIDUE INTERACTION FILTER
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:04:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

• Trajectory scan complete. (Safely bypassed 1 un-docked anomalies).

🏆 ABSOLUTE MECHANISTIC POCKET LEADERBOARD (TOP 5 LEADS)
-------------------------------------------------------------------------------------
         Identifier  Vina_Energy  CNN_Score  CNN_Affinity  Proximity_Å
      CID_134495310     -9.45308   0.255970      5.751024        2.271
CID_134495307_iso_1     -9.45308   0.255970      5.751024        2.271
CID_169603487_iso_1     -9.40715   0.793882      6.489149        2.418
CID_121304065_iso_1     -9.32984   0.189164      5.562547        1.841
CID_137641304_iso_1     -9.32578   0.380674      6.040578        2.054
-------------------------------------------------------------------------------------

🎉 Success! Top 5 mechanism-specific complexes exported to: /content/drive/MyDrive/GNINA_Receptor3_Project/Mechanism_Specific_Top_5


[23:04:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.


In [5]:
import os
import math
import pandas as pd
from rdkit import Chem
from google.colab import drive

print("==========================================================")
print("📊 SCRIPT 2: LIGAND EXTRACTION BY CNN POSE CONFIDENCE")
print("==========================================================")

try: drive.mount('/content/drive')
except Exception: pass

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/Top_5_Percent_CNN"

RECEPTOR_PATH = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"

if not os.path.exists(CHECKPOINT_DIR):
    raise FileNotFoundError(f"❌ Checkpoint directory missing at: {CHECKPOINT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_poses = []
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for f_path in docked_files:
    supp = Chem.SDMolSupplier(f_path)
    for mol in supp:
        if mol is None: continue
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(all_poses)}"
        all_poses.append({
            "mol_obj": mol,
            "Identifier": name,
            "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
            "CNN_Score": float(mol.GetProp('CNNscore')),
            "CNN_Affinity": float(mol.GetProp('CNNaffinity'))
        })

df = pd.DataFrame(all_poses)
# Sort descending: values closest to 1.0 represents absolute structural probability
df_sorted = df.sort_values(by="CNN_Score", ascending=False).reset_index(drop=True)

cutoff = max(1, int(math.ceil(len(df_sorted) * 0.05)))
top_cnn = df_sorted.head(cutoff)

print(f"\n🏆 TOP 5% CNN POSE SCORE LEADERBOARD (Isolating {cutoff} out of {len(df_sorted)} targets)")
print("-" * 70)
print(top_cnn[["Identifier", "Vina_Energy", "CNN_Score"]].to_string(index=False))
print("-" * 70)

with open(RECEPTOR_PATH, 'r') as f:
    receptor_text = f.read()

for idx, row in top_cnn.iterrows():
    clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
    out_pdb = f"{OUTPUT_DIR}/CNN_Rank_{idx+1}_{clean_id}_complex.pdb"
    with open(out_pdb, 'w') as out:
        out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

print(f"\n🎉 Success! Top 5% CNN complexes exported to: {OUTPUT_DIR}")

📊 SCRIPT 2: LIGAND EXTRACTION BY CNN POSE CONFIDENCE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:08:51] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

KeyError: 'minimizedAffinity'

In [6]:
import os
import math
import pandas as pd
from rdkit import Chem
from google.colab import drive

print("==========================================================")
print("📊 Patched SCRIPT 2: LIGAND EXTRACTION BY CNN POSE CONFIDENCE")
print("==========================================================")

# Mount Google Drive
try: drive.mount('/content/drive')
except Exception: pass

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/Top_5_Percent_CNN"

RECEPTOR_PATH = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"

if not os.path.exists(CHECKPOINT_DIR):
    raise FileNotFoundError(f"❌ Checkpoint directory missing at: {CHECKPOINT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_poses = []
skipped_count = 0
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for f_path in docked_files:
    supp = Chem.SDMolSupplier(f_path)
    for mol in supp:
        if mol is None: continue

        # 🛡️ PROTECTIVE LAYER: Silently skip any ligand records missing GNINA scoring tags
        if not mol.HasProp('minimizedAffinity') or not mol.HasProp('CNNscore'):
            skipped_count += 1
            continue

        name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(all_poses)}"
        all_poses.append({
            "mol_obj": mol,
            "Identifier": name,
            "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
            "CNN_Score": float(mol.GetProp('CNNscore')),
            "CNN_Affinity": float(mol.GetProp('CNNaffinity'))
        })

print(f"• File aggregation complete. (Safely bypassed {skipped_count} un-docked anomalies).")

df = pd.DataFrame(all_poses)
# Sort descending: values closest to 1.0 represent premium geometric shape alignment
df_sorted = df.sort_values(by="CNN_Score", ascending=False).reset_index(drop=True)

cutoff = max(1, int(math.ceil(len(df_sorted) * 0.05)))
top_cnn = df_sorted.head(cutoff)

print(f"\n🏆 TOP 5% CNN POSE SCORE LEADERBOARD (Isolating {cutoff} out of {len(df_sorted)} targets)")
print("-" * 70)
print(top_cnn[["Identifier", "Vina_Energy", "CNN_Score"]].to_string(index=False))
print("-" * 70)

with open(RECEPTOR_PATH, 'r') as f:
    receptor_text = f.read()

for idx, row in top_cnn.iterrows():
    clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
    out_pdb = f"{OUTPUT_DIR}/CNN_Rank_{idx+1}_{clean_id}_complex.pdb"
    with open(out_pdb, 'w') as out:
        out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

print(f"\n🎉 Success! Top 5% CNN geometric complexes exported to: {OUTPUT_DIR}")

📊 Patched SCRIPT 2: LIGAND EXTRACTION BY CNN POSE CONFIDENCE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
• File aggregation complete. (Safely bypassed 1 un-docked anomalies).

🏆 TOP 5% CNN POSE SCORE LEADERBOARD (Isolating 15 out of 289 targets)
----------------------------------------------------------------------
         Identifier  Vina_Energy  CNN_Score
CID_130312738_iso_1     -6.60190   0.865297
CID_170217542_iso_1     -7.44810   0.817678
      CID_121310013     -6.12562   0.811064
CID_121304236_iso_1     -6.12562   0.811064
CID_169603487_iso_1     -9.40715   0.793882
      CID_156835675     -6.20208   0.787013
 CID_56832850_iso_1     -5.43724   0.784873
      CID_121304052     -6.73081   0.780927
CID_121304227_iso_1     -6.73081   0.780927
CID_121304226_iso_1     -6.73081   0.780927
CID_170150503_iso_1     -6.90504   0.778924
CID_138486537_iso_1     -6.81400   0.771484
CID_135356546_iso_1     -7

[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:10:55] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol


🎉 Success! Top 5% CNN geometric complexes exported to: /content/drive/MyDrive/GNINA_Receptor3_Project/Top_5_Percent_CNN


In [7]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from google.colab import drive

print("==========================================================")
print("🔍 SCRIPT 3: MECHANISM-SPECIFIC RESIDUE INTERACTION FILTER")
print("==========================================================")

try: drive.mount('/content/drive')
except Exception: pass

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/Mechanism_Specific_Top_5"

RECEPTOR_PATH = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"
TARGET_RESIDUES_STR = "resid 41 80 108 119 120 134"
CONTACT_CUTOFF = 4.0

try:
    import MDAnalysis as mda
except ImportError:
    import subprocess
    subprocess.run("pip install -q rdkit mdanalysis pandas", shell=True)
    import MDAnalysis as mda

if not os.path.exists(CHECKPOINT_DIR):
    raise FileNotFoundError(f"❌ Checkpoint directory missing at: {CHECKPOINT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

u_rec = mda.Universe(RECEPTOR_PATH)
pocket_atoms = u_rec.select_atoms(TARGET_RESIDUES_STR)
target_coords = pocket_atoms.positions

valid_binders = []
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for f_path in docked_files:
    supp = Chem.SDMolSupplier(f_path)
    for mol in supp:
        if mol is None: continue
        conf = mol.GetConformer()
        ligand_coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
        distances = np.linalg.norm(ligand_coords[:, np.newaxis, :] - target_coords[np.newaxis, :, :], axis=2)
        min_dist = np.min(distances)

        if min_dist <= CONTACT_CUTOFF:
            name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(valid_binders)}"
            valid_binders.append({
                "mol_obj": mol,
                "Identifier": name,
                "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
                "CNN_Score": float(mol.GetProp('CNNscore')),
                "CNN_Affinity": float(mol.GetProp('CNNaffinity')),
                "Proximity_Å": round(min_dist, 3)
            })

if len(valid_binders) == 0:
    print("❌ Selection Matrix Failure: No compounds located within 4.0 Å of active site side chains.")
else:
    df_filtered = pd.DataFrame(valid_binders)
    df_sorted = df_filtered.sort_values(by="Vina_Energy", ascending=True).reset_index(drop=True)
    top_5_leads = df_sorted.head(5)

    print("\n🏆 ABSOLUTE MECHANISTIC POCKET LEADERBOARD (TOP 5 LEADS)")
    print("-" * 85)
    print(top_5_leads[["Identifier", "Vina_Energy", "CNN_Score", "CNN_Affinity", "Proximity_Å"]].to_string(index=False))
    print("-" * 85)

    with open(RECEPTOR_PATH, 'r') as f:
        receptor_text = f.read()

    for idx, row in top_5_leads.iterrows():
        clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
        out_pdb = f"{OUTPUT_DIR}/Absolute_Mechanistic_Rank_{idx+1}_{clean_id}_complex.pdb"
        with open(out_pdb, 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

    print(f"\n🎉 Success! Top 5 mechanism-specific complexes exported to: {OUTPUT_DIR}")

🔍 SCRIPT 3: MECHANISM-SPECIFIC RESIDUE INTERACTION FILTER
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:17:47] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

KeyError: 'minimizedAffinity'

In [8]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from google.colab import drive

print("==========================================================")
print("🔍 PATCHED SCRIPT 3: RESIDUE INTERACTION FILTER CORE")
print("==========================================================")

# Mount Google Drive
try: drive.mount('/content/drive')
except Exception: pass

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"
OUTPUT_DIR = f"{DRIVE_PROJECT_DIR}/Mechanism_Specific_Top_5"

RECEPTOR_PATH = f"{DRIVE_PROJECT_DIR}/receptor3.pdb"
TARGET_RESIDUES_STR = "resid 41 80 108 119 120 134"
CONTACT_CUTOFF = 4.0

try:
    import MDAnalysis as mda
except ImportError:
    import subprocess
    subprocess.run("pip install -q rdkit mdanalysis pandas", shell=True)
    import MDAnalysis as mda

if not os.path.exists(CHECKPOINT_DIR):
    raise FileNotFoundError(f"❌ Checkpoint directory missing at: {CHECKPOINT_DIR}")
os.makedirs(OUTPUT_DIR, exist_ok=True)

u_rec = mda.Universe(RECEPTOR_PATH)
pocket_atoms = u_rec.select_atoms(TARGET_RESIDUES_STR)
target_coords = pocket_atoms.positions

valid_binders = []
skipped_count = 0
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

for f_path in docked_files:
    supp = Chem.SDMolSupplier(f_path)
    for mol in supp:
        if mol is None: continue

        # 🛡️ PROTECTIVE LAYER: Silently bypass any records lacking structural docking data
        if not mol.HasProp('minimizedAffinity') or not mol.HasProp('CNNscore'):
            skipped_count += 1
            continue

        conf = mol.GetConformer()
        ligand_coords = np.array([list(conf.GetAtomPosition(i)) for i in range(mol.GetNumAtoms())])
        distances = np.linalg.norm(ligand_coords[:, np.newaxis, :] - target_coords[np.newaxis, :, :], axis=2)
        min_dist = np.min(distances)

        if min_dist <= CONTACT_CUTOFF:
            name = mol.GetProp("_Name") if mol.HasProp("_Name") else f"Compound_ID_{len(valid_binders)}"
            valid_binders.append({
                "mol_obj": mol,
                "Identifier": name,
                "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
                "CNN_Score": float(mol.GetProp('CNNscore')),
                "CNN_Affinity": float(mol.GetProp('CNNaffinity')),
                "Proximity_Å": round(min_dist, 3)
            })

print(f"• Spatial proximity analysis complete. (Safely bypassed {skipped_count} un-docked entries).")

if len(valid_binders) == 0:
    print("❌ Selection Matrix Failure: Zero compounds located within 4.0 Å of active site side chains.")
else:
    df_filtered = pd.DataFrame(valid_binders)
    df_sorted = df_filtered.sort_values(by="Vina_Energy", ascending=True).reset_index(drop=True)
    top_5_leads = df_sorted.head(5)

    print("\n🏆 ABSOLUTE MECHANISTIC POCKET LEADERBOARD (TOP 5 LEADS)")
    print("-" * 85)
    print(top_5_leads[["Identifier", "Vina_Energy", "CNN_Score", "CNN_Affinity", "Proximity_Å"]].to_string(index=False))
    print("-" * 85)

    with open(RECEPTOR_PATH, 'r') as f:
        receptor_text = f.read()

    for idx, row in top_5_leads.iterrows():
        clean_id = "".join([c if c.isalnum() else "_" for c in row["Identifier"]])
        out_pdb = f"{OUTPUT_DIR}/Absolute_Mechanistic_Rank_{idx+1}_{clean_id}_complex.pdb"
        with open(out_pdb, 'w') as out:
            out.write(receptor_text + "\n" + Chem.MolToPDBBlock(row["mol_obj"]))

    print(f"🎉 Success! Absolute Top 5 pocket complexes safely synchronized to: {OUTPUT_DIR}")

🔍 PATCHED SCRIPT 3: RESIDUE INTERACTION FILTER CORE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:04] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

• Spatial proximity analysis complete. (Safely bypassed 1 un-docked entries).

🏆 ABSOLUTE MECHANISTIC POCKET LEADERBOARD (TOP 5 LEADS)
-------------------------------------------------------------------------------------
         Identifier  Vina_Energy  CNN_Score  CNN_Affinity  Proximity_Å
      CID_134495310     -9.45308   0.255970      5.751024        2.271
CID_134495307_iso_1     -9.45308   0.255970      5.751024        2.271
CID_169603487_iso_1     -9.40715   0.793882      6.489149        2.418
CID_121304065_iso_1     -9.32984   0.189164      5.562547        1.841
CID_137641304_iso_1     -9.32578   0.380674      6.040578        2.054
-------------------------------------------------------------------------------------
🎉 Success! Absolute Top 5 pocket complexes safely synchronized to: /content/drive/MyDrive/GNINA_Receptor3_Project/Mechanism_Specific_Top_5


[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:19:05] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

In [9]:
import os
import pandas as pd
from rdkit import Chem
from google.colab import drive

print("==========================================================")
print("📝 SCRIPT: AUGMENTING LEADERBOARD WITH CANONICAL SMILES")
print("==========================================================")

# Mount Google Drive
try: drive.mount('/content/drive')
except Exception: pass

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/GNINA_Receptor3_Project"
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints_remdsim"

# Define the exact 5 elite identifiers captured on your leaderboard
TARGET_IDS = [
    "CID_134495310",
    "CID_134495307_iso_1",
    "CID_169603487_iso_1",
    "CID_121304065_iso_1",
    "CID_137641304_iso_1"
]

if not os.path.exists(CHECKPOINT_DIR):
    raise FileNotFoundError(f"❌ Checkpoint directory missing at: {CHECKPOINT_DIR}")

matched_data = {}
docked_files = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith("_docked.sdf")]

print("• Parsing trajectories to map high-fidelity 2D strings...")
for f_path in docked_files:
    supp = Chem.SDMolSupplier(f_path)
    for mol in supp:
        if mol is None: continue
        name = mol.GetProp("_Name") if mol.HasProp("_Name") else ""

        # Isolate target molecules and translate structural graphs to SMILES
        if name in TARGET_IDS and name not in matched_data:
            canonical_smiles = Chem.MolToSmiles(mol, canonical=True)
            matched_data[name] = {
                "Identifier": name,
                "Vina_Energy": float(mol.GetProp('minimizedAffinity')),
                "CNN_Score": float(mol.GetProp('CNNscore')),
                "CNN_Affinity": float(mol.GetProp('CNNaffinity')),
                "SMILES": canonical_smiles
            }

# Reconstruct data frame to mirror your absolute ranking history
ordered_rows = [matched_data[tid] for tid in TARGET_IDS if tid in matched_data]
df_smiles = pd.DataFrame(ordered_rows)

print("\n==========================================================================================")
print(" 🏆 FINAL EXTRACTED MECHANISTIC LEADERBOARD WITH CANONICAL SMILES")
print("==========================================================================================")
pd.set_option('display.max_colwidth', None)
print(df_smiles[["Identifier", "Vina_Energy", "CNN_Score", "SMILES"]].to_string(index=False))
print("==========================================================================================")

📝 SCRIPT: AUGMENTING LEADERBOARD WITH CANONICAL SMILES
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
• Parsing trajectories to map high-fidelity 2D strings...

 🏆 FINAL EXTRACTED MECHANISTIC LEADERBOARD WITH CANONICAL SMILES
         Identifier  Vina_Energy  CNN_Score                                                                                          SMILES
      CID_134495310     -9.45308   0.255970          CC(C)(N[P@](=O)(OC[C@H]1O[C@@](C#N)(c2ccc3c(N)ncnn23)[C@H](O)[C@@H]1O)Oc1ccccc1)C(=O)O
CID_134495307_iso_1     -9.45308   0.255970          CC(C)(N[P@](=O)(OC[C@H]1O[C@@](C#N)(c2ccc3c(N)ncnn23)[C@H](O)[C@@H]1O)Oc1ccccc1)C(=O)O
CID_169603487_iso_1     -9.40715   0.793882 C[C@H](N[P@](=O)(OC[C@@H]1O[C@@](C)(c2ccc3c(N)ncnn23)[C@H](O)[C@@H]1O)Oc1ccccc1)C(=O)OCC(C)(C)C
CID_121304065_iso_1     -9.32984   0.189164     CC(C)OC(=O)C(C)(C)N[P@](=O)(OC[C@H]1O[C@@](C#N)(c2ccc3c(N)ncnn23)[C@H](O)[C@@H]1O)Oc

[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[23:21:02] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol